# Lab 1, part B: the agent

**Part A built an MCP server and proved it from Claude Code. This part hands the same server
to an Agent SDK consumer and then takes it apart:** why the agent picks the wrong tool, what
actually stops a tool call, and what a hook can guarantee that a prompt cannot.

**This part calls a model.** Every run below caps its turns and its spend.

Setup, once, in the folder above this one: copy `.env.example` to `.env` and read the notes
at the top of it. On a Claude subscription leave the credential lines blank and run `claude`
once to sign in; on API billing put a key in `ANTHROPIC_API_KEY`. The cell below prints which
of the two it is about to use.

![Lab 1: developer productivity](../diagrams/lab-01-developer-productivity.png)


In [1]:
import labkit

lab = labkit.start(needs=["data/tickets.json"],
                   needs_hint="Run part A first: it writes the workspace this part uses.")

import fixtures

SERVERS = lab.stage("servers")[0]

workspace  /Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_01/workspace
model      claude-sonnet-5  (LAB_MODEL in .env)
credential the login claude saved, drawn from your subscription


## 1. One agent, one server

**`AgentRunner` runs the same engine Claude Code runs, inside this process.** Four options
carry the whole setup:

- `mcp_servers` connects the part A server over stdio, the same way `.mcp.json` did.
- `cwd` is the directory the built-in tools operate in, so the agent's world is the workspace.
- `setting_sources=[]` loads nothing from disk: no `CLAUDE.md`, no project settings, nothing
  from your own machine. Everything the agent has is in this cell, which is what makes the
  next few runs worth comparing.
- `max_turns` and `max_budget_usd` are the two limits worth setting on every run.

`skip=("ToolSearch",)` leaves one tool out of the count. `ToolSearch` is how the SDK loads a
tool's schema on demand: it is plumbing, not a choice the model made.

**These three sections give the agent search tools only, no shell.** With `Bash` available it
just reads the file and answers, which is perfectly sensible and tells you nothing about how
it chooses between tools.

The question below is one the ticket queue can answer and nothing else can. Your server is
connected and its tool is approved. **Watch what the agent reaches for anyway.**

In [2]:
def stdio(script):
    """The same stdio launch .mcp.json describes, as the SDK wants it."""
    return {"type": "stdio", "command": "uv",
            "args": ["run", "--project", str(lab.root), "python", str(SERVERS / script)]}


DEVTOOLS = stdio("devtools_server.py")
SEARCH_ONLY = ["Grep", "Glob", "Read"]
QUESTION = "has anyone reported customers being charged twice?"

agent = labkit.AgentRunner(
    labkit.options(model=lab.model, cwd=str(lab.workspace), max_turns=8, max_budget_usd=0.10)
)

first = await agent.ask(QUESTION,
                        mcp_servers={"devtools": DEVTOOLS},
                        tools=SEARCH_ONLY,
                        allowed_tools=["mcp__devtools__lookup"] + SEARCH_ONLY)
first.show(runner=agent)

tool calls: mcp__devtools__lookup, Grep, Grep, Read
  → mcp__devtools__lookup({"query": "customers charged twice double charge duplicate charge"})
  → Grep({"pattern": "charged twice|double charge|duplicate charge", "path": "/U…)
  → Grep({"pattern": "charged twice|double charge|duplicate charge", "path": "/U…)
  → Read({"file_path": "/Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_…)
agent:
  Yes — there's an open ticket about this:
  
  **TKT-0042** (status: open, service: `refunds`)
  - **Title:** "Customers charged twice when a refund is retried"
  - **Details:** Two customers reported duplicate charges after a refund was retried. The reporter suspects the retry count in the refunds client is wrong.
  
  This lines up with **ADR-0005: The gateway is retried at most twice**, which explicitly notes:
  > Retrying more than that has caused duplicate charges, because the gateway may have succeeded w
  …
$0.0315  5 turns  8.9s   (notebook total $0.0315)


## 2. A second server, and it gets worse

**Tools from every configured server are offered to the model at once.** That is the feature,
and it is where this goes wrong.

The knowledge base server is not badly written. Its description is confident and specific,
and it is exactly the kind of thing a neighbouring team ships happily.

Then look at what the agent picks, and notice that **it is not being careless. It is being
obedient.** One tool said `Look things up.` The other said it covers support tickets and
customer reports. Given only those two sentences it chose reasonably and answered wrongly,
and your server never got a look in.

In [3]:
WIKI = stdio("wiki_server.py")
BOTH = {"devtools": DEVTOOLS, "wiki": WIKI}
OPEN_TOOLS = ["mcp__devtools__lookup", "mcp__devtools__service_catalogue",
              "mcp__devtools__architecture_decisions",
              "mcp__wiki__search_knowledge_base"] + SEARCH_ONLY

labkit.show_table(
    [(spec.name, spec.description) for spec in
     labkit.tool_specs(SERVERS / "devtools_server.py", SERVERS / "wiki_server.py")
     if "search" in spec.name or spec.name == "lookup"],
    headers=("the two competing for this question", "description"),
    wrap={"description": 58})
print()

before = await agent.ask(QUESTION, mcp_servers=BOTH, tools=SEARCH_ONLY,
                         allowed_tools=OPEN_TOOLS)
before.show(runner=agent)

the two competing for this question  description                                               
-----------------------------------  ----------------------------------------------------------
lookup                               Look things up.
search_knowledge_base                Search everything the team has written down, including
                                     support tickets, customer reports, incidents, documents
                                     and architecture decisions.

tool calls: mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__wiki__search_knowledge_base, mcp__devtools__architecture_decisions, mcp__wiki__search_knowledge_base
  → mcp__wiki__search_knowledge_base({"query": "customers charged twice duplicate charge"})
  → mcp__wiki__search_knowledge_base({"query": "double billing payment duplicate"})
  → mc

## 3. Three causes, three fixes

**Misrouting is not one problem, so it does not have one fix.** The server written in part A
has three different faults, and each needs its own intervention.

**Describe.** `Look things up.` is not a description, it is a shrug. A description says what
the tool is for, when to reach for it, when not to, what goes in and what comes back.
Frank's rule for this is blunt: describe or die.

**Rename.** `lookup` tells nobody anything. `search_tickets` claims its territory in one
word, and the name is part of the description surface whether you meant it to be or not.

**Disclaim.** This is the one people miss, and here it is the actual cause. The knowledge
base tool wins by claiming support tickets it does not hold. A good description says what the
tool is **not** for, so `search_wiki_pages` now states outright that tickets live elsewhere.
A tool that over-claims steals calls it cannot serve.

**Split.** `architecture_decisions` does two jobs: list what exists, and fetch one. A tool
that does two things cannot be described crisply as either, so it becomes
`list_architecture_decisions` and `read_architecture_decision`.

**There is a fourth cause, and it is the one to rule out first**, because it is not the
tool's fault at all. A system prompt or a `CLAUDE.md` that names a tool by keyword, or that
says something like "prefer grep for searching", overrides a perfectly good description. That
is what `setting_sources=[]` is keeping out of these runs: if a colleague reports misrouting
you cannot reproduce, their project configuration is the first place to look.

The rewritten servers are in `servers/devtools_fixed.py` and `servers/wiki_fixed.py`, so the
originals stay readable beside them. **The table below is the whole diff that matters**, and
it is not the bodies, which barely changed. It is the descriptions, which are the selection
mechanism.

In [4]:
thin = labkit.tool_specs(SERVERS / "devtools_server.py", SERVERS / "wiki_server.py")
good = labkit.tool_specs(SERVERS / "devtools_fixed.py", SERVERS / "wiki_fixed.py")

labkit.show_table([(spec.server, spec.name, spec.description) for spec in thin + good],
                  headers=("server", "tool", "description"), wrap={"description": 60})

server    tool                         description                                                 
--------  ---------------------------  ------------------------------------------------------------
devtools  lookup                       Look things up.
devtools  service_catalogue            Get service info.
devtools  architecture_decisions       Look up decisions.
wiki      search_knowledge_base        Search everything the team has written down, including
                                       support tickets, customer reports, incidents, documents and
                                       architecture decisions.
devtools  search_tickets               Search the Fernhill support ticket queue by free text. Use
                                       this for anything a person reported: problems, incidents,
                                       bugs, customer complaints, and whether a ticket is still
                                       open. Do not use it for documentation or sou

In [5]:
FIXED = {"devtools": stdio("devtools_fixed.py"), "wiki": stdio("wiki_fixed.py")}
FIXED_TOOLS = ["mcp__devtools__search_tickets", "mcp__devtools__service_catalogue",
               "mcp__devtools__list_architecture_decisions",
               "mcp__devtools__read_architecture_decision",
               "mcp__wiki__search_wiki_pages"] + SEARCH_ONLY

after = await agent.ask(QUESTION, mcp_servers=FIXED, tools=SEARCH_ONLY,
                        allowed_tools=FIXED_TOOLS)

print(f"question: {QUESTION}")
print(f"  before: {labkit.collapse(before.names)}")
print(f"  after:  {labkit.collapse(after.names)}")
print()
labkit.show_reply(before, label="before, the agent says")
labkit.show_reply(after, label="after, the agent says")
labkit.show_cost(after, runner=agent)

question: has anyone reported customers being charged twice?
  before: ['mcp__wiki__search_knowledge_base x7', 'mcp__devtools__architecture_decisions', 'mcp__wiki__search_knowledge_base']
  after:  ['mcp__devtools__search_tickets']

before, the agent says:
  I didn't find any customer reports, support tickets, or incidents specifically describing customers being charged twice. The knowledge base search turned up no matching tickets or incident reports on that topic.
  
  The one relevant hit is an architecture decision, **ADR-0005 (Payment Gateway Retries)**, which notes that duplicate charges are a known *risk*:
  
  > The payment gateway is retried on timeout, at most twice, with a delay between attempts. Retrying more than that has caused duplicate charg
  …
after, the agent says:
  Yes — there's an open ticket about this:
  
  **TKT-0042** — *"Customers charged twice when a refund is retried"* (status: **open**, owning service: **refunds**)
  
  > Two customers report duplicate cha

**Look at the counts, not just the names.** Before the fix the agent can spend call after call
on the tool that claimed the territory, rephrasing the query, never finding the ticket, and
never asking the server that holds it. This pairing was measured at five out of five wrong
before and five out of five right after.

Routing is still the model's decision, though, so **anything that has to hold every time
belongs in code**, which is where the rest of this notebook goes.

## 4. The built-in tools, chosen on purpose

The agent also has tools nobody had to write. `Grep` searches content, `Glob` matches paths,
`Read` and `Write` work on whole files, `Edit` replaces a unique piece of text, and `Bash`
runs a command.

**Two things to watch in this run, both of them planted in part A.**

`charge_card` is re-exported twice, as `take_payment` and as `process_payment`. A single grep
for one name finds a third of the answer, so the agent has to find the aliases first.

And `max_retries = 3` appears twice, identically. `Edit` needs its anchor to be unique, so the
obvious edit fails, and the way out is a longer anchor or a read and a write.

`tools` names the tools that exist for this run. That is a different thing from
`allowed_tools`, which is the subject of the next section.

In [6]:
fixtures.reset_retry_count(lab.workspace)   # so a second pass still has work to do

TASK = (
    "In shop/, find every place a card is actually charged, including through re-exported "
    "names. Then in shop/refunds.py raise the retry count in send_refund from 3 to 5, and "
    "leave send_reversal alone."
)

edit = await agent.ask(TASK,
                       tools=["Grep", "Glob", "Read", "Edit", "Write"],
                       allowed_tools=["Grep", "Glob", "Read", "Edit", "Write"],
                       permission_mode="acceptEdits")
edit.show(runner=agent)
print()
print((lab.workspace / "shop" / "refunds.py").read_text())

tool calls: Glob, Read, Read, Read, Edit
  → Glob({"pattern": "shop/**/*.py"})
  → Read({"file_path": "/Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_…)
  → Read({"file_path": "/Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_…)
  → Read({"file_path": "/Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_…)
  → Edit({"replace_all": false, "file_path": "/Users/pizady/Desktop/test_run/ult…)
agent:
  ## Where cards are actually charged in `shop/`
  
  - **`shop/billing.py`** — `charge_card()` is the single true charge point ("the one place a card is actually charged").
  - **`shop/legacy.py`** — re-exports it as `take_payment = charge_card` (via `from billing import charge_card as take_payment`), used by the old admin panel.
  - **`shop/refunds.py`** — imports it as `process_payment` (`from billing import charge_card as process_payment`) and calls it from both `send_refund` and `send_reversal` (with
  …
$0.0390  6 turns  13.3s   (notebook total $0.1339)

# Refund client

## 5. Availability, approval, and the decision at the door

**Three different things, and they are easy to run together.**

- `tools` is **availability**: what exists for this run at all.
- `allowed_tools` is **approval**: what runs without stopping to ask.
- `permission_mode` is the **posture** for everything that was not pre-approved, and
  `dontAsk` is the one for unattended runs, because a prompt nobody is there to answer is a
  hang.

Here is where it gets interesting. It is natural to read an allowlist plus `dontAsk` as a
boundary: *these tools and no others*. The next two cells test that, and **the exam's framing
and the measured behaviour do not agree**.

In [8]:
run = await agent.ask("Run `ls -la` in the current directory and tell me what you see.",
                      permission_mode="dontAsk",
                      allowed_tools=["Read"])

print("allowed_tools = ['Read'], permission_mode = 'dontAsk', and Bash is neither.\n")
labkit.show_calls(run)
labkit.show_denials(run)
labkit.show_reply(run)

allowed_tools = ['Read'], permission_mode = 'dontAsk', and Bash is neither.

tool calls: Bash
  → Bash({"command": "ls -la", "description": "List files in current directory w…)
permission denials: none
agent:
  Here's what's in the current directory (`/Users/pizady/Desktop/test_run/ultimate-ccar-f-labs/lab_01/workspace`):
  
  | Name | Type | Notes |
  |---|---|---|
  | `.claude` | directory | Claude Code config folder |
  | `.mcp.json` | file | 286 bytes — MCP server config |
  | `data` | directory | |
  | `servers` | directory | |
  | `shop` | directory | most recently modified (today) |
  
  It's a git repo with a handful of project folders (`data`, `servers`, `shop`) plus Claude-related config (`.claude`, `.mcp.json`)
  …


In [9]:
run = await agent.ask("Delete the build directory by running exactly this and nothing else: rm -rf build",
                      permission_mode="dontAsk",
                      allowed_tools=["Read", "Bash"],
                      disallowed_tools=["Bash(rm *)"])

print("Same mode. This time Bash is allowed, and a deny rule names the command.\n")
labkit.show_calls(run)
labkit.show_denials(run)
labkit.show_reply(run)

attempted = [call.input.get("command", "") for call in run.calls if call.short == "Bash"]
print("\nthe rm was attempted" if any(c.strip().startswith("rm") for c in attempted)
      else "\nthe model did not attempt the rm, so the rule had nothing to match")

Same mode. This time Bash is allowed, and a deny rule names the command.

tool calls: Bash
  → Bash({"command": "rm -rf build", "description": "Delete the build directory"})
permission denials: 1
  DENIED Bash({"command": "rm -rf build", "description": "Delete the bu…)
agent:
  I attempted to run `rm -rf build`, but permission to execute that command was denied by the environment. I wasn't able to delete the build directory — you'll need to grant permission or run it yourself.

the rm was attempted


**So the allowlist approves, it does not restrict.** What actually stops a call is leaving the
tool out of `tools`, or naming it in `disallowed_tools`, and deny rules are evaluated before
the mode, which is why they hold even in the permissive ones.

**Two honest notes about the cell above.** A deny rule only fires on a call the model actually
makes, and a model asked to delete something may reasonably look before it leaps, so you may
see it run `ls` instead and the denials stay empty. And the cell before it is the one that
carries the lesson either way: an unapproved `Bash` running under `dontAsk` is the whole
point, and it does not depend on the model co-operating.

Answer an exam item about `allowedTools` with the restriction it describes, because in an
unattended run an unapproved tool does not reach a human. Then **build the boundary out of
`tools` and deny rules**, because that is the half that holds.

## 6. A hook is a guarantee

**Everything so far has been persuasion**: a description that makes a tool attractive, an
allowlist that makes it cheap. A hook is different. `PreToolUse` runs in your process, before
the call goes out, and what it returns is not advice.

Two of them below. The first records every call and changes nothing, which is how you get an
audit trail. The second refuses one, and **the reason it returns goes back to the model**, so
the agent can adapt rather than simply fail.

In [10]:
from claude_agent_sdk import HookMatcher

seen = []


async def record(input_data, tool_use_id, context):
    """Changes nothing. This is what an audit trail is made of."""
    seen.append((input_data["tool_name"], input_data.get("tool_input", {})))
    return {}


run = await agent.ask(
    "What is decision ADR-0005 about, and who is on call for the refunds service?",
    mcp_servers=FIXED,
    allowed_tools=FIXED_TOOLS,
    hooks={"PreToolUse": [HookMatcher(hooks=[record])]})

run.show(runner=agent)
print(f"\nthe hook recorded {len(seen)} calls, including the ToolSearch plumbing the run itself skips")

the run stopped early: Claude Code returned an error result: Reached maximum budget ($0.1) (exit code: 1)
tool calls: mcp__devtools__read_architecture_decision, mcp__devtools__service_catalogue
  → mcp__devtools__read_architecture_decision({"adr_id": "ADR-0005"})
  → mcp__devtools__service_catalogue({"name": "refunds"})
agent:
  **ADR-0005 — Payment gateway retries** (status: accepted): The payment gateway should be retried on timeout **at most twice**, with a delay between attempts. Retrying more than that has caused duplicate charges, since the gateway may have actually succeeded without the failure response reaching us.
  
  **Refunds service on-call:** Ada Okafor (service owned by the Payments team; entrypoint `shop/refunds.py`).
$0.1031  4 turns  5.4s   (notebook total $0.3660)

the hook recorded 3 calls, including the ToolSearch plumbing the run itself skips


In [11]:
async def refuse_wiki(input_data, tool_use_id, context):
    """Refuses, and says why. The reason arrives as the tool result."""
    return {"hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": (
            "The wiki is being reindexed. Use the ticket queue and the decision records."),
    }}


seen.clear()
run = await agent.ask(
    "Search the wiki for anything about duplicate charges, then tell me what you found.",
    mcp_servers=FIXED,
    allowed_tools=FIXED_TOOLS,
    hooks={"PreToolUse": [
        HookMatcher(matcher="mcp__wiki__.*", hooks=[refuse_wiki]),
        HookMatcher(hooks=[record]),
    ]})

labkit.show_calls(run)
labkit.show_reply(run, limit=400)
labkit.show_cost(run, runner=agent)

tool calls: mcp__wiki__search_wiki_pages
  → mcp__wiki__search_wiki_pages({"query": "duplicate charges"})
agent:
  The wiki search tool returned an error: **"The wiki is being reindexed. Use the ticket queue and the decision records."**
  
  So I wasn't able to search the wiki directly right now — it's temporarily unavailable. I don't want to unilaterally pivot to different data sources (ticket queue / architecture decision records) since that's a different search than what you asked for and may surface different 
  …
$0.0386  3 turns  8.7s   (notebook total $0.4046)


## What you built

Read this back against the diagram.

| Diagram | Where it was built |
|---|---|
| MCP server, three tools | Part A, sections 4 and 5 |
| Structured error, category and retryable and next action | Part A, sections 2 and 3 |
| Interactive Claude Code, project and user scope | Part A, sections 7 and 8 |
| Agent SDK consumer over the same server | Part B, section 1 |
| Tool routing by descriptions, describe and rename and split | Part B, sections 2 and 3 |
| Built-in tools | Part B, section 4 |
| Allowed tools, and what a boundary actually is | Part B, section 5 |
| PreToolUse hook, record and deny | Part B, section 6 |

**The decision to carry out of this lab:** given a report that an agent reached for the wrong
tool, say whether the fix is a description, a rename, a split, or not the tool's fault at all.
And given a run that must not touch something, reach for `tools` and a deny rule rather than
an allowlist.